# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# metadata is a Metadata object, we use its documented attributes
print(f"Dataset title: {dataset.metadata.name}\n\n{dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's inspect all record sets defined in this dataset, their fields, and the associated `@id` values.

In [ ]:
# List all record sets and fields, showing their @id
print("Record sets in this dataset:")
for record_set in dataset.metadata.record_sets:
    print(f"- Record Set Name: {record_set.name}, @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field: {field.name}, @id: {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here we extract all main record sets and load them into pandas DataFrames. This lets you preview table columns and their contents.

In [ ]:
# Extract data from each record set using @id
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns of the first record set as an example (replace with actual @id as desired)
if record_sets:
    example_record_set = record_sets[0]
    print(f"Record set @id: {example_record_set}")
    print("Columns:", dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Select a numeric field from a chosen record set
- Filter records by threshold value
- Normalize the selected field
- Group data by a categorical field if available

In [ ]:
# Example EDA on the first record set (replace IDs as required)
if record_sets:
    record_set_id = record_sets[0]
    df = dataframes[record_set_id]
    # Find a numeric field candidate
    numeric_field_id = None
    for field in dataset.metadata.record_set(record_set_id).fields:
        if field.data_type in ["Integer", "Float", "Number"]:
            numeric_field_id = field.id
            break
    if numeric_field_id is not None and numeric_field_id in df.columns:
        # Replace threshold as needed
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold} (mean value):")
        display(filtered_df.head())
        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try to group by a non-numeric field
        group_field = None
        for field in dataset.metadata.record_set(record_set_id).fields:
            if field.data_type not in ["Integer", "Float", "Number"]:
                group_field = field.id
                break
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No non-numeric field found suitable for grouping.")
    else:
        print("No numeric field found in this record set.")
else:
    print("No record sets loaded for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll provide a boxplot of the selected numeric field as an example.

In [ ]:
import matplotlib.pyplot as plt

if record_sets and 'filtered_df' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    filtered_df.boxplot(column=numeric_field_id)
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print("No valid numeric field for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, inspect, and perform basic analysis on the FAIR\^2 Mass Spectrometry dataset using Croissant metadata references. Using the `mlcroissant` library, you can programmatically query record sets and fields by their `@id`, ensuring robust and reproducible data workflows.

Next steps might include:
- Deeper domain analysis (e.g., comparing protein abundances between conditions)
- Integration with other omics data or annotation sources
- Custom visualizations for publication-ready figures

For more information on Croissant datasets and `mlcroissant`, visit: https://github.com/mlcommons/croissant